# Lab 009 — Robust vs. Naïve Standard Errors

**Lesson:** [`0009-regression-done-right.html`](../lessons/0009-regression-done-right.html)

**The one skill:** run a regression of overlapping returns on a signal, watch a gaudy naïve
t-statistic, then **deflate it honestly** with heteroskedasticity-robust (White) and
heteroskedasticity-and-autocorrelation-consistent (**Newey–West / HAC**) standard errors —
and confirm by hand that OLS is projection (residuals orthogonal to `X`) and that
`β̂ = Cov(x,y)/Var(x)`.

**Exit criteria:** the EXIT TICKET cell prints cleanly (all CHECK cells pass).

**How this notebook works**

| Cell tag | You do |
|----------|--------|
| **PROVIDED** | Run it. Imports, data, helpers. |
| **TODO** | Fill the `____` blanks. This is where the learning is. |
| **CHECK** | Run it — immediate assertions. Don't edit. |
| **EXIT TICKET** | Final deliverable. Prints your findings. |

**Environment:** Python 3 + `numpy` + `statsmodels`. Fully self-contained (simulated data — no
network needed). See [`labs/README.md`](./README.md).

### Running on Google Colab?

Colab opens only this single file, so the lab dependencies (numpy, scipy, statsmodels, …) and
the course repo are **not** guaranteed to be present. The cell below fixes that: on Colab it
shallow-clones the course repo, installs `requirements-labs.txt`, and switches into `labs/` so
relative paths resolve. **On a local venv or Binder it does nothing — just run it and continue.**

In [ ]:
# @colab-bootstrap — PROVIDED. Makes the lab self-sufficient on Google Colab; a no-op elsewhere.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/financial-engineering"):
        !git clone --depth 1 https://github.com/Avistian/financial-engineering.git /content/financial-engineering
    %pip install -q -r /content/financial-engineering/requirements-labs.txt
    os.chdir("/content/financial-engineering/labs")
    print("Colab ready — working dir:", os.getcwd())
else:
    print("Not on Colab — using the local environment as-is.")

## Concept recap (read before coding)

**OLS.** With design matrix $X$ ($n\times p$, first column of 1s for the intercept) and outcome $y$,
$$\hat\beta = (X^\top X)^{-1}X^\top y,\qquad \text{one predictor: } \hat\beta_1=\frac{\operatorname{Cov}(x,y)}{\operatorname{Var}(x)}=\frac{S_{xy}}{S_{xx}}.$$

**Projection.** Fitted $\hat y = Hy$ with hat matrix $H=X(X^\top X)^{-1}X^\top$; residuals $e=(I-H)y$ are **orthogonal to every column of $X$**, so $\sum_i e_i=0$ and $\sum_i x_i e_i=0$ (a checkable identity).

**Inference.** Classical $\operatorname{Var}(\hat\beta)=\sigma^2 (X^\top X)^{-1}$ assumes constant variance and no autocorrelation; $t=\hat\beta_1/\operatorname{SE}(\hat\beta_1)$.

**When it breaks (both leave $\hat\beta$ unbiased — only the SE is wrong):**
- **Heteroskedasticity** → **White / HC** SE (each residual's own $e_i^2$). `cov_type="HC1"`.
- **Autocorrelation / overlapping returns** → **Newey–West (HAC)** SE (Bartlett-weighted residual autocovariances out to $L$ lags). `cov_type="HAC", maxlags=L`.

**Overlap.** Sampling an $h$-day forward return every day makes consecutive errors share $h-1$ days → strong positive autocorrelation → naïve SE far too small → inflated $t$.

In [ ]:
# PROVIDED — a self-contained simulation where we KNOW the truth.
# A persistent daily signal weakly predicts daily returns (heteroskedastic noise). We then form
# OVERLAPPING h-day forward returns sampled every day, regressed on the signal. Consecutive
# observations share h-1 days, so the regression errors are strongly autocorrelated — exactly the
# setting where a naïve t-stat lies.
import numpy as np
import statsmodels.api as sm

rng = np.random.default_rng(9)
n = 800          # number of overlapping observations
h = 21           # forward horizon in trading days (~1 month)

# Persistent (AR(1)) standardized daily signal.
sig = np.zeros(n + h + 1)
for t in range(1, n + h + 1):
    sig[t] = 0.95 * sig[t-1] + rng.normal(0, 1)
sig = (sig - sig.mean()) / sig.std()

b_daily = 0.02                                   # tiny TRUE daily predictive slope
vol = 0.01 * (1 + 0.5 * np.abs(sig))             # heteroskedastic daily vol
daily_ret = b_daily * sig + vol * rng.normal(0, 1, size=n + h + 1)

# Overlapping h-day FORWARD return aligned to the signal at time t.
y = np.array([daily_ret[t+1:t+1+h].sum() for t in range(n)])   # outcome
x = sig[:n]                                                    # predictor
print(f"n = {n} overlapping obs, horizon h = {h} days")
print(f"signal lag-1 autocorrelation: {np.corrcoef(x[:-1], x[1:])[0,1]:.3f}  (persistent)")

### Task 1 — OLS by hand, and confirm it is a projection
**Goal:** compute the slope from the covariance identity and verify the residuals are orthogonal to `x`.
Fill `Sxx` and `Sxy`; the rest is provided.

**Why it matters:** the orthogonality $\sum e_i=0,\ \sum x_i e_i=0$ *is* "OLS is projection" — the residual has no linear info left about the predictor.

**Hint boundary:** `Sxx = np.sum((x-xbar)**2)`, `Sxy = np.sum((x-xbar)*(y-ybar))`.

In [ ]:
# TODO — OLS slope/intercept from the covariance identity.
xbar, ybar = x.mean(), y.mean()
Sxx = ____                       # np.sum((x - xbar)**2)
Sxy = ____                       # np.sum((x - xbar)*(y - ybar))
beta1 = Sxy / Sxx
beta0 = ybar - beta1 * xbar
resid = y - (beta0 + beta1 * x)
print(f"beta1 (Cov/Var) = {beta1:.4f},  beta0 = {beta0:.4f}")
print(f"sum(resid) = {resid.sum():.2e},  sum(x*resid) = {np.sum(x*resid):.2e}  (both ~0 => residual ⟂ X)")

In [ ]:
# CHECK — do not edit
assert abs(resid.sum()) < 1e-6, "residuals must sum to zero (orthogonal to the constant)"
assert abs(np.sum(x * resid)) < 1e-6, "residuals must be orthogonal to x (the projection property)"
X = sm.add_constant(x)
_ols = sm.OLS(y, X).fit()
assert np.allclose([beta0, beta1], _ols.params, atol=1e-9), "hand OLS must match statsmodels"
print("Task 1 OK — β̂ = Cov/Var matches statsmodels, and residuals ⟂ X (OLS is projection).")

### Task 2 — Classical vs. White (heteroskedasticity-robust) standard errors
**Goal:** fit OLS (classical SEs) and get the **White HC1** version. Store `se_white` (the slope's robust SE).

**Why it matters:** White uses each residual's own $e_i^2$ instead of one pooled $\hat\sigma^2$, so it survives non-constant variance. (Here the *autocorrelation* in Task 3 dominates, so White ≈ classical — a lesson in matching the fix to the violation.)

**Hint boundary:** `fit_white = fit.get_robustcov_results(cov_type="HC1")`; `se_white = fit_white.bse[1]`.

In [ ]:
# TODO — classical and heteroskedasticity-robust (White) SEs.
X = sm.add_constant(x)
fit = sm.OLS(y, X).fit()                       # classical (nonrobust) SEs
se_classical, t_classical = fit.bse[1], fit.tvalues[1]

fit_white = fit.get_robustcov_results(cov_type="HC1")
se_white = ____                                # fit_white.bse[1]
t_white = fit_white.tvalues[1]
print(f"classical : SE = {se_classical:.4f},  t = {t_classical:.2f}")
print(f"White HC1 : SE = {se_white:.4f},  t = {t_white:.2f}")

In [ ]:
# CHECK — do not edit
assert se_white > 0 and np.isfinite(t_white), "White SE/t must be finite and positive"
assert abs(se_white - se_classical) / se_classical < 0.25, "here White ≈ classical: overlap, not heteroskedasticity, is the issue"
print("Task 2 OK — White SE computed; barely moves the t here, so the real problem is elsewhere (Task 3).")

### Task 3 — Newey–West (HAC): the fix for overlapping returns
**Goal:** recompute the slope's SE with a **HAC** estimator using `maxlags = h` (the overlap horizon). Store `se_nw`.

**Why it matters:** overlapping windows make the errors strongly positively autocorrelated, so the naïve SE is far too small. Newey–West adds Bartlett-weighted residual autocovariances out to `L` lags and restores the honest, larger SE — deflating the t-stat.

**Hint boundary:** `fit_nw = fit.get_robustcov_results(cov_type="HAC", maxlags=h, use_correction=True)`; `se_nw = fit_nw.bse[1]`.

In [ ]:
# TODO — heteroskedasticity-AND-autocorrelation-consistent (Newey–West) SE.
fit_nw = fit.get_robustcov_results(cov_type="HAC", maxlags=h, use_correction=True)
se_nw = ____                                   # fit_nw.bse[1]
t_nw = fit_nw.tvalues[1]
print(f"classical  : SE = {se_classical:.4f},  t = {t_classical:.2f}")
print(f"Newey–West : SE = {se_nw:.4f},  t = {t_nw:.2f}   (maxlags = {h})")
print(f"the naïve t was inflated by {t_classical / t_nw:.1f}x")

In [ ]:
# CHECK — do not edit
assert se_nw > 1.5 * se_classical, "with h-day overlap the HAC SE should be much larger than the naïve one"
assert abs(t_classical) > abs(t_nw), "the naïve t-stat must deflate once autocorrelation is accounted for"
print("Task 3 OK — Newey–West SE >> classical; the inflated naïve t-stat deflates to an honest one.")

### Task 4 — How many *independent* observations did you really have?
**Goal:** turn the SE inflation into an **effective sample size**. Store `inflation = se_nw/se_classical` and `n_eff = n / inflation**2`.

**Why it matters:** a variance inflated by a factor means your $n$ overlapping points were worth only $n_\text{eff}$ independent ones — the concrete cost of overlap.

**Hint boundary:** `inflation = se_nw / se_classical`; `n_eff = n / inflation**2`.

In [ ]:
# TODO — effective sample size implied by the SE inflation.
inflation = ____                               # se_nw / se_classical
n_eff = n / inflation**2
print(f"SE inflation (nw / classical) = {inflation:.2f}x")
print(f"effective independent observations ≈ {n_eff:.0f}  (out of {n} overlapping ones)")

In [ ]:
# CHECK — do not edit
assert inflation > 1.0, "positive autocorrelation must inflate the honest SE"
assert n_eff < n, "overlap means fewer independent observations than the raw count"
print(f"Task 4 OK — {n} overlapping obs were worth only ≈ {n_eff:.0f} independent ones.")

In [ ]:
# EXIT TICKET — paste this output to your teacher.
print("=== Robust vs naïve standard errors ===")
print(f"Regression                    : {h}-day overlapping fwd return  ~  signal   (n = {n})")
print(f"Slope β̂ = Cov/Var             : {beta1:.4f}   (unbiased under both violations)")
print(f"Residual ⟂ X (projection)     : sum e = {resid.sum():.1e},  sum x·e = {np.sum(x*resid):.1e}")
print(f"t-stat  classical / White / NW: {t_classical:.2f} / {t_white:.2f} / {t_nw:.2f}")
print(f"SE inflation (NW / classical) : {inflation:.2f}x   ->  n_eff ≈ {n_eff:.0f}")
print()
print("One-sentence takeaway (edit me):")
print(f"Overlapping returns faked a t of {t_classical:.0f}; the honest Newey–West t is {t_nw:.0f}, because the")
print(f"{n} observations carried only ~{n_eff:.0f} independent pieces of information — the point estimate")
print("was fine, but the naïve standard error (not the slope) was the lie.")

### Stretch (optional, ungraded)
- **Vary the lag count.** Recompute the Newey–West SE for `maxlags` in `{0, 5, 21, 42, 63}`. At `maxlags=0` you recover White; watch the SE climb and plateau once `L` spans the overlap `h`. If your "significance" is fragile to `L`, distrust it.
- **Non-overlapping baseline.** Rebuild `y` sampling every `h`-th day (no overlap) and rerun: the classical and Newey–West SEs should now agree, and `n_eff ≈ n`. This is the cleanest fix of all — don't overlap if you can avoid it.
- **Spurious regression (trap 3).** Regress one random-walk *level* on another independent random-walk level (`np.cumsum` of noise). Watch a huge R² and |t| > 4 appear between two series with nothing in common — then confirm it vanishes when you regress their *differences* (returns).
- **Heteroskedasticity, isolated.** Drop the overlap (`h=1`) but crank the vol's dependence on `|signal|`; now White should pull away from classical while Newey–West adds little — the mirror image of this lab.